# Hands-on 30 (Bab 12): Pipeline Lengkap Sensor -> MQTT -> Python -> Prediksi -> Storage -> Dashboard

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 12.

Pipeline ujung ke ujung Smart Room Monitoring: simulator mengirim telemetry ke MQTT,
subscriber Python melakukan preprocessing (lag + moving average dari buffer), model
klasifikasi memberi status (normal/warning/fault), hasil disimpan. Di buku storage
memakai InfluxDB + dashboard Grafana; di sini didemokan dengan storage CSV lokal
(InfluxDB opsional via `--influx`, butuh Docker).

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Jalankan Pipeline End-to-End (broker + subscriber + simulator)

In [2]:
import subprocess, sys, time

broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = subprocess.Popen(
        [sys.executable, "scripts/pipeline_room_subscriber.py",
         "--max-messages", "8", "--timeout", "40"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(1)
    pub = subprocess.run([sys.executable, "scripts/publisher_room_simulator.py",
                          "--count", "8", "--interval", "0.3"],
                         capture_output=True, text=True)
    print(pub.stdout)
    out, _ = sub.communicate()
    print("--- Output subscriber pipeline ---")
    print(out)
finally:
    broker.terminate()

Published: {"timestamp": "2026-10-22T08:07:39", "device_id": "room01", "temperature_c": 30.88, "humidity_pct": 41.68, "co2_ppm": 1316, "power_watt": 285.29}
Published: {"timestamp": "2026-10-22T08:07:40", "device_id": "room01", "temperature_c": 27.79, "humidity_pct": 66.71, "co2_ppm": 583, "power_watt": 195.68}
Published: {"timestamp": "2026-10-22T08:07:40", "device_id": "room01", "temperature_c": 30.46, "humidity_pct": 62.73, "co2_ppm": 1476, "power_watt": 282.31}
Published: {"timestamp": "2026-10-22T08:07:40", "device_id": "room01", "temperature_c": 25.75, "humidity_pct": 43.85, "co2_ppm": 770, "power_watt": 235.91}
Published: {"timestamp": "2026-10-22T08:07:41", "device_id": "room01", "temperature_c": 30.68, "humidity_pct": 61.44, "co2_ppm": 487, "power_watt": 251.53}
Published: {"timestamp": "2026-10-22T08:07:41", "device_id": "room01", "temperature_c": 33.3, "humidity_pct": 62.53, "co2_ppm": 996, "power_watt": 220.22}
Published: {"timestamp": "2026-10-22T08:07:41", "device_id": "r

## 2. Verifikasi Hasil di Storage CSV

In [3]:
import pandas as pd
df = pd.read_csv("outputs/reports/pipeline_room_predictions.csv")
print(df.to_string(index=False))
print("\nDistribusi status:", df["predicted_status"].value_counts().to_dict())

          timestamp device_id  temperature_c  humidity_pct  co2_ppm  power_watt predicted_status
2026-10-22T08:07:39    room01          30.88         41.68     1316      285.29            fault
2026-10-22T08:07:40    room01          27.79         66.71      583      195.68           normal
2026-10-22T08:07:40    room01          30.46         62.73     1476      282.31            fault
2026-10-22T08:07:40    room01          25.75         43.85      770      235.91           normal
2026-10-22T08:07:41    room01          30.68         61.44      487      251.53          warning
2026-10-22T08:07:41    room01          33.30         62.53      996      220.22          warning
2026-10-22T08:07:41    room01          26.95         48.15     1047      243.73           normal
2026-10-22T08:07:42    room01          26.50         45.39      736      273.28           normal

Distribusi status: {'normal': 4, 'fault': 2, 'warning': 2}


## Penutup

Seluruh komponen menyatu: MQTT menjadi pintu masuk pipeline analitik, subscriber
Python menjadi pusat logika (validasi, fitur lag/rolling dari buffer), model dipanggil
otomatis tanpa notebook, dan hasil inferensi tersimpan. Untuk deployment penuh sesuai
buku, jalankan InfluxDB 2 dan Grafana via Docker, aktifkan `--influx`, lalu susun panel
suhu/kelembaban/CO2/daya/status dengan ambang warna normal-hijau, warning-kuning,
fault-merah.